In [ ]:
import torch
import torch.nn as nn 
import torch.nn.functional as F
import torch.optim as optim
import random

class TwoTower(nn.Module):
    def __init__(self, user_num, item_num, emb_dim=256):
        super().__init__()
        self.user_emb = nn.Embedding(user_num, emb_dim)
        self.item_emb = nn.Embedding(item_num, emb_dim)

        self.user_nn = nn.Linear(emb_dim, 64)
        self.item_nn = nn.Linear(emb_dim, 64)

        self.temperature = nn.Parameter(torch.tensor(0.05))
        self.relu = nn.ReLU()
        self.norm_user = nn.BatchNorm1d(64)
        self.norm_item = nn.BatchNorm1d(64)
        self.dropout = nn.Dropout(0.5)

    def forward(self, users, items):
        user_embeddings = self.user_emb(users.squeeze(-1))
        user_rep = self.relu(self.user_nn(user_embeddings))
        user_rep = self.norm_user(user_rep)
        # user_rep = self.dropout(user_rep)

        user_rep = F.normalize(user_rep, p=2, dim=1)


        item_embeddings = self.item_emb(items.squeeze(-1))
        item_rep = self.relu(self.item_nn(item_embeddings))
        item_rep = self.norm_item(item_rep)
        # item_rep = self.dropout(item_rep)

        item_rep = F.normalize(item_rep, p=2, dim=1)
        
        sim = torch.matmul(user_rep, item_rep.transpose(0, 1))
        sim = sim / self.temperature

        return sim

NUM_USERS = 50000
NUM_ITEMS = 100000
BATCH_SIZE = 256
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

user_data = torch.randint(1,NUM_USERS,(10000,1)).to(device)
item_data = torch.randint(1,NUM_ITEMS,(10000,1)).to(device)

def get_batches(users, items, batch_size, shuffle=True):
    n_samples = users.size(0)
    indices = torch.arange(n_samples)
    if shuffle:
        indices = torch.randperm(n_samples)

    for start_idx in range(0, n_samples, batch_size):
        end_idx = min(start_idx + batch_size, n_samples)

        if end_idx - start_idx < 2:
            continue

        batch_idx = indices[start_idx:end_idx]

        yield users[batch_idx], items[batch_idx]


def train():
    EPOCH_NUM = 500
    LR = 0.0001
    model = TwoTower(NUM_USERS, NUM_ITEMS, emb_dim=128).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-3)
    loss_func = nn.CrossEntropyLoss().to(device)
    model.train()
    min_loss = 10
    for epoch in range(EPOCH_NUM):
        optimizer.zero_grad()
        total_loss = 0.0
        for batch_u, batch_i in get_batches(user_data, item_data, BATCH_SIZE):

            logits = model(batch_u, batch_i)
            target = torch.arange(batch_u.size(0)).long().to(device)
            loss = loss_func(logits, target)

            loss.backward()

            optimizer.step()
            total_loss += loss.item()

        min_loss = min(min_loss, total_loss)
        if (epoch + 1) % 10 == 0:
            print(f'epoch: {epoch + 1} / {EPOCH_NUM}, loss: {total_loss/BATCH_SIZE:.6f}')

    print(f'min loss: {min_loss}')

train()


epoch: 10 / 500, loss: 0.843058
epoch: 20 / 500, loss: 0.691363
epoch: 30 / 500, loss: 0.586090
epoch: 40 / 500, loss: 0.467285
epoch: 50 / 500, loss: 0.314929
epoch: 60 / 500, loss: 0.171310
epoch: 70 / 500, loss: 0.084358
epoch: 80 / 500, loss: 0.044792
epoch: 90 / 500, loss: 0.029716
epoch: 100 / 500, loss: 0.024275
epoch: 110 / 500, loss: 0.018518
epoch: 120 / 500, loss: 0.019245
epoch: 130 / 500, loss: 0.015256
epoch: 140 / 500, loss: 0.012038
epoch: 150 / 500, loss: 0.010297
epoch: 160 / 500, loss: 0.009875
epoch: 170 / 500, loss: 0.010590
epoch: 180 / 500, loss: 0.009048
epoch: 190 / 500, loss: 0.009331
epoch: 200 / 500, loss: 0.008849
epoch: 210 / 500, loss: 0.008262
epoch: 220 / 500, loss: 0.006702
epoch: 230 / 500, loss: 0.006821
epoch: 240 / 500, loss: 0.005872
epoch: 250 / 500, loss: 0.006202
epoch: 260 / 500, loss: 0.006394
epoch: 270 / 500, loss: 0.006992
epoch: 280 / 500, loss: 0.005207
epoch: 290 / 500, loss: 0.004934
epoch: 300 / 500, loss: 0.004542
epoch: 310 / 500, l